In [0]:
%run
Utils/configuration

imblearn is installed

Out[3]: DataFrame[]

In [0]:
import numpy as np



In [0]:
# dados - lista de tuplas [(),(),()]
valores = [
  ('Patricia','F','IDOSO','BRANCO',10,'1'),
  ('Helena','F','CRIANCA','BRANCO',50,'1'),
  ('Cristina','F','ADOLESCENTE','PARDO',60,'1'),
  ('Isabela','F','ADULTO','NEGRO',70,'1'),
  ('Jorge','M','IDOSO','MAMELUCO',3,'1'),
  ('Paulo','M','ADULTO','BRANCO',74,'1'),
  ('Arthur','M', 'ADULTO','INDIGENA',100,'1' ),
  ('Manuela','F','IDOSO','BRANCO',13,'1'),
  ('Hanna','F','CRIANCA','BRANCO',15,'0'),
  ('Alice','F','ADOLESCENTE','PARDO',18,'0'),
  ('Aluska','F','ADULTO','NEGRO',21,'0'),
  ('Miguel','M','IDOSO','MAMELUCO',23,'0'),
  ('Lana','F','ADULTO','BRANCO',74,'0'),
  ('Luiz','M', 'ADULTO','INDIGENA',90,'0' )
  ]

# colunas - lista []

colunas = ["Nome",
   "SEXO"
  ,"FAIXA_ETARIA"
  ,"ETNIA"
  ,"IDADE"
  ,"Y"
  ]

df = spark.createDataFrame(valores, colunas)

categoria = ["SEXO", "FAIXA_ETARIA","ETNIA"]

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import itertools

In [0]:

def Amostra_Populacional(df, categoria, qtde_amostra):
  
  '''
    df_pd = DataFrame do tipo Spark
    categoria = lista das variáveis categóricas para a extração da amostra
    qtde_amostra = 
  '''
  df_pd = df.toPandas()
  df_retorno = []
  lista = []
  produto_list = []
  ls_cat = df_pd[categoria].astype('category')
  for i in categoria:
    lista.append(df.select(i).distinct().count())
  vl_amostra = (np.prod(lista)) #* qtde_amostra
  for i in lista:
    produto_list.append(int(vl_amostra/i))
  cat = []
  for i in ls_cat:
    cat.append(list(df_pd[i].unique()))
  combinacoes = np.multiply(produto_list, cat)
  combinacoes = list(combinacoes)
  combinacoes[0].sort()
  combinacoes = np.array(combinacoes)
  combinacoes = np.transpose(combinacoes)
  #df_retorno = pd.DataFrame(combinacoes, columns=categoria)
  igualdades = []
  for lista in combinacoes[0:len(combinacoes)]:
    for dados,col in itertools.zip_longest(lista, categoria):
      dados = f'{col} == "{dados}"'
      igualdades.append(dados)
  igualdades = np.array(igualdades)
  #igualdades = igualdades.reshape(vl_amostra,3)
  #separador = ' AND '
  #condicao_filtro = []
  #for i in igualdades[0:len(igualdades)]:
  #  condicao_filtro.append([separador.join(i)])
  #separador2 = ','
  #condicao_filtro2 = []
  #for i in condicao_filtro:
  #  condicao_filtro2.append(separador2.join(i))
  #empty = SparkSession.builder.appName('Empty_Dataframe').getOrCreate()
  #emp_RDD = spark.sparkContext.emptyRDD()
  #filtro3 = empty.createDataFrame(emp_RDD, df.schema)
  #for i in condicao_filtro2[1:len(condicao_filtro2)]:
  #  filtro2 = df.filter(i).limit(qtde_amostra)
  #  filtro3 = filtro2.union(filtro3)
  return igualdades

In [0]:
teste = Amostra_Populacional(df, categoria, 1)
display(teste)

Nome,SEXO,FAIXA_ETARIA,ETNIA,IDADE,Y
Arthur,M,ADULTO,INDIGENA,100,1
Paulo,M,ADULTO,BRANCO,74,1
Miguel,M,IDOSO,MAMELUCO,23,0
Lana,F,ADULTO,BRANCO,74,0
Isabela,F,ADULTO,NEGRO,70,1
Alice,F,ADOLESCENTE,PARDO,18,0
Hanna,F,CRIANCA,BRANCO,15,0


#Validação com dados da base

In [0]:
dev_usr = dbutils.secrets.get(scope="see-cmsp-dw", key="integracao-dev-usr")
dev_pwd = dbutils.secrets.get(scope="see-cmsp-dw", key="integracao-dev-pwd")

dev_url, dev_prop = get_db_cfg(db_name= 'db_sara_dev', usr= dev_usr, pwd=dev_pwd)

In [0]:
query = """(SELECT NR_RG
                  ,CD_ESCOL
                  ,ID_TURNO
                  ,ID_GRAU
                  ,ID_SERIE
                  ,ID_TURMA
                  ,DT_INIC_MATRIC
                  ,DT_FIM_MATRIC
                  ,DT_NASC
                  ,FL_SEXO 
              FROM CADALUNOS.dbo.TB_MATRICULA) AMOSTRA """
df_amostra = spark.read.jdbc(url = dev_url, table = query, properties = dev_prop)

In [0]:
display(df_amostra)

In [0]:
df_amostra.write\
.option("overwriteSchema", True) \
.mode("overwrite") \
.format("delta") \
.option("numPartitions", 16)\
.option("lowerBound", 8)\
.option("upperBound", 16)\
.partitionBy(["ID_TURNO","ID_GRAU", "ID_SERIE", "FL_SEXO"]) \
.save(f"{path_bronze}teste_amostra/df_amostra")

In [0]:
df_amostra = spark.read.format("delta").load(f"{path_bronze}teste_amostra/df_amostra")

In [0]:
display(df_amostra.limit(10))

NR_RG,CD_ESCOL,ID_TURNO,ID_GRAU,ID_SERIE,ID_TURMA,DT_INIC_MATRIC,DT_FIM_MATRIC,DT_NASC,FL_SEXO
000106585331,0217724,1,01,02,A,20110207,20111223,20030610,F
000108717884,0217724,1,01,02,A,20110207,20110323,20030512,F
000104940837,0217724,1,01,02,A,20110207,20110323,20030127,F
000106107959,0217724,1,01,02,A,20110207,20111223,20030109,F
000106307574,0217724,1,01,02,A,20110207,20111223,20030119,F
000104959723,0217724,1,01,02,B,20110207,20111223,20030327,F
000106305896,0217724,1,01,02,B,20110207,20111223,20030407,F
000103160693,0217724,1,01,02,B,20110207,20111223,20030619,F
000104879596,0082417,1,01,02,B,20110207,20111223,20030414,F
000108118159,0227997,1,01,02,A,20110207,20111223,20030314,F


In [0]:
categoricas = ['ID_TURNO', 'ID_GRAU', 'ID_SERIE', 'FL_SEXO']

In [0]:
teste2 = Amostra_Populacional(df_amostra, categoricas, 100)
display(teste2)

--------------------------------------------------------------------------- 
 ValueError Traceback (most recent call last)
 <command-130163773725356> in <module> 
 ----> 1 teste2 = Amostra_Populacional ( df_amostra , categoricas , 100 ) 
 2 display ( teste2 ) 

 <command-1513522336282433> in Amostra_Populacional (df, categoria, qtde_amostra) 
 31 igualdades . append ( dados ) 
 32 igualdades = np . array ( igualdades ) 
 ---> 33 igualdades = igualdades . reshape ( vl_amostra , 3 ) 
 34 separador = ' AND ' 
 35 condicao_filtro = [ ] 

 ValueError : cannot reshape array of size 18000 into shape (4500,3)